# MagBridge-Battery v1.0 — full generation run

This notebook produces the final v1.0 release dataset.

**Expected runtime:** ~5–6 hours on Colab free tier (CPU only).
**Output:** ~6,760 samples (5,600 clean + 600 paired anomalies + 560 regime-B), packaged as Parquet shards + manifest + splits.

## What you need before starting

1. **The project zip** (`MagBridge-Battery-Phase2.zip` or similar). This contains:
   - `src/magbridge/` — all bridge code
   - `data/v1.0/` — locked NPZ artifacts + PulseBat CSV
   - `configs/generation_config.yaml` — locked v1.0 config

2. **A Google Drive folder** (optional, recommended) where the output will be saved. If Colab disconnects mid-run, you lose the progress — but if Drive is mounted, the *next* run's output goes there.

## What this notebook does, step by step

1. Installs Python dependencies (`pyarrow`, `pydantic`, `pennylane`, etc.)
2. Uploads or unzips the project archive
3. Confirms the fast test suite passes (~80s sanity check)
4. Runs the full v1.0 generation pipeline (~5–6 hours)
5. Zips the output into one downloadable bundle
6. Either downloads to your machine or saves to Drive

**Determinism:** Same inputs + same code + same seed → byte-identical output. The output includes a manifest with hashes of all inputs and the config, so a future user can verify they have exactly this dataset.


## 1. Install dependencies

In [ ]:
!pip install -q 'pydantic>=2' 'pyarrow>=15' 'pyyaml>=6' 'pennylane>=0.40' pandas scikit-learn pytest

## 2. Upload the project archive

Choose ONE of the two paths below. The first uses Colab's file upload widget; the second mounts Google Drive (recommended if you have the project there).

### Option A — Upload the zip directly (one-time)

In [ ]:
# Uncomment this cell if you want to upload from your local machine
# from google.colab import files
# uploaded = files.upload()  # pick MagBridge-Battery-Phase2.zip

### Option B — Mount Google Drive (recommended)

Put the project zip in your Drive at e.g. `MyDrive/Magridge/MagBridge-Battery-Phase2.zip`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Adjust these paths to match where you put the zip in Drive
import os, shutil

DRIVE_ZIP = '/content/drive/MyDrive/MagBridge-Battery/magbridge_inputs.zip'
LOCAL_ZIP = '/content/magbridge_inputs.zip'

if os.path.exists(DRIVE_ZIP):
    shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
    print(f'Copied from Drive: {DRIVE_ZIP} -> {LOCAL_ZIP}')
else:
    print(f'Drive path not found: {DRIVE_ZIP}')
    print('Either fix the path above or use Option A (file upload) instead.')

In [ ]:
# Unpack the project
import zipfile, os
WORK_DIR = '/content/magbridge_work'
os.makedirs(WORK_DIR, exist_ok=True)

with zipfile.ZipFile(LOCAL_ZIP) as z:
    z.extractall(WORK_DIR)

# Find the actual project root (might be nested under a folder named like the zip)
import glob
candidates = glob.glob(f'{WORK_DIR}/*/src/magbridge') + glob.glob(f'{WORK_DIR}/src/magbridge')
assert candidates, f'Could not find src/magbridge under {WORK_DIR}; check the zip structure'
PROJECT_ROOT = os.path.dirname(os.path.dirname(candidates[0]))
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print()
print('Contents:')
for entry in sorted(os.listdir(PROJECT_ROOT)):
    print(f'  {entry}')

## 3. Sanity check: run the fast test suite

This takes ~80 seconds. If anything fails here, stop — the generation run won't succeed either.

In [ ]:
import subprocess, sys
env = os.environ.copy()
env['PYTHONPATH'] = f"{PROJECT_ROOT}/src"
env['PYTHONUNBUFFERED'] = '1'   # force unbuffered output
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '--tb=short', '-q'],
    cwd=PROJECT_ROOT, env=env, capture_output=True, text=True,
)
print('STDOUT:'); print(result.stdout[-3000:])
print('STDERR:'); print(result.stderr[-1500:])
assert result.returncode == 0, 'Test suite failed; do not proceed with generation.'

## 4. Run the full generation pipeline

This is the long step (~5–6 hours). Progress prints every 100 samples.

**If Colab disconnects:** generation is not resumable — you'll need to rerun from scratch. Output is fully deterministic, so a re-run produces the same dataset bit-for-bit.


In [ ]:
OUTPUT_DIR = '/content/magbridge_output_v1_0'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Run the generator as a module so logging works cleanly.
# The --bridge-code-commit flag stamps the manifest with a release tag
# (set this to whatever you want; the default falls back to NOT_IN_GIT).
import subprocess, sys
env = os.environ.copy()
env['PYTHONPATH'] = f"{PROJECT_ROOT}/src"
env['PYTHONUNBUFFERED'] = '1'   # force unbuffered output
cmd = [
    sys.executable, '-u', '-m', 'magbridge.generate',
    '--config', f'{PROJECT_ROOT}/configs/generation_config.yaml',
    '--data-dir', f'{PROJECT_ROOT}/data/v1.0',
    '--output-dir', OUTPUT_DIR,
    '--bridge-code-commit', 'MAGBRIDGE_V1_0_PHASE2_FIXED_20260515',
]
print('Running:', ' '.join(cmd))
# Use Popen so we get streaming output, not buffered
import time
t0 = time.time()
proc = subprocess.Popen(cmd, cwd=PROJECT_ROOT, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
elapsed = time.time() - t0
print(f'\n=== Generation done in {elapsed/60:.1f} min ===')
assert proc.returncode == 0, 'Generation failed; inspect output above.'

## 5. Inspect the output

Quick check that the manifest looks right and the Parquet shards are well-formed.

In [ ]:
import json, pyarrow.parquet as pq, pandas as pd
from pathlib import Path

out = Path(OUTPUT_DIR)
print('=== Output structure ===')
for path in sorted(out.rglob('*')):
    if path.is_file():
        size_kb = path.stat().st_size / 1024
        print(f'  {path.relative_to(out)}  ({size_kb:.1f} KB)')

print()
print('=== Manifest summary ===')
with open(out / 'manifest.json') as f:
    m = json.load(f)
print(f"  Dataset:           {m['dataset_name']} v{m['dataset_version']}")
print(f"  Total samples:     {m['n_total_samples']}")
print(f"    clean grounded:    {m['n_clean_grounded_samples']}")
print(f"    synthetic anomaly: {m['n_synthetic_anomaly_samples']}")
print(f"    regime-B:          {m['n_regime_b_extrapolation_samples']}")
print(f"  Bridge:            {m['bridge_version']}")
print(f"  OSF hash:          {m['osf_data_hash']}")
print(f"  PulseBat hash:     {m['pulsebat_data_hash']}")
print(f"  Config hash:       {m['config_hash']}")
print(f"  Commit:            {m['bridge_code_commit']}")
print(f"  Generated UTC:     {m['generated_at_utc']}")

print()
print('=== Sample-level inspection (first shard) ===')
df = pq.read_table(out / 'data' / 'shard_0000.parquet').to_pandas()
print(f'  Shape: {df.shape}')
print(f'  Columns: {list(df.columns)[:8]}...')
print(f'  Per-category counts in shard_0000:')
print(df['anomaly_origin'].value_counts().to_string())

## 6. Bundle and download

Pack everything into one zip and either download to your machine, or save to Drive.

In [ ]:
import shutil, datetime as _dt

stamp = _dt.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
bundle_name = f'magbridge_battery_v1_0_{stamp}.zip'
bundle_path = f'/content/{bundle_name}'
shutil.make_archive(bundle_path.replace('.zip', ''), 'zip', root_dir=OUTPUT_DIR)
print(f'Bundle ready: {bundle_path}')
import os
print(f'Size: {os.path.getsize(bundle_path) / (1024*1024):.1f} MB')

In [ ]:
# Option 1: download to your local machine
# from google.colab import files
# files.download(bundle_path)

In [ ]:
# Option 2: save to Drive (recommended for ~5 hours of compute)
DRIVE_DEST = '/content/drive/MyDrive/MagBridge-Battery/'
os.makedirs(DRIVE_DEST, exist_ok=True)
shutil.copy(bundle_path, DRIVE_DEST)
print(f'Saved to Drive: {DRIVE_DEST}{os.path.basename(bundle_path)}')

## Done

Your downloaded zip contains:

```
manifest.json                         # provenance: hashes, config, citation
splits/
  by_cell_primary.json                # recommended split (39/8/9 cells)
  by_record_optimistic_baseline.json  # leakage-prone, for ablation only
data/
  metadata.parquet                    # labels only (no signals); quick to load
  shard_0000.parquet ... 0004.parquet # full samples with signal columns
```

Next steps (outside this notebook):
- Use the bundle as the v1.0 release artifact
- Upload to Zenodo to mint a DOI
- Push to Hugging Face Datasets for `load_dataset()` access
- Update the manifest citation block with the Zenodo DOI before publishing
